# Develop

## Import

In [1]:
import numpy as np
import pandas as pd
import json

import pyarrow as pa
import pyarrow.parquet as pq

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

from catboost import CatBoostClassifier, Pool
from sklearn.metrics import average_precision_score

from typing import List

## Model learning

In [2]:
def train_in_chunks(files_list: List[str] = None,
                    cols_json_file: str = None,
                    batchsize: int = 20000,
                    model: CatBoostClassifier = None,
                    auto_nan_filler: bool = True):
    """
    Функция для последовательного обучения модели по чанкам. Принимает следующие аргументы:
    files_list - список с файлами для обучения модели (полные пути до файлов)
    cols_json_file - путь до json файла с инфой по колонкам
    batchsize - размер чанка (кол-во строк)
    model - модель
    """
    if files_list is None:
        raise AttributeError("Надо указать files_list - файл(ы) для обучения. Обязательно в списке")
    if cols_json_file is None:
        raise AttributeError("Надо указать cols_json_file - json файл с инфой по колонкам")
    if model is None:
        raise AttributeError("Надо указать model - действующую модель")
    
    with open(cols_json_file, 'r') as file:
        cols = json.load(file)
    
    feature_cols = cols['all']
    cat_cols = cols['cat']
    target_col = cols['target']
    
    first_batch_flag = True
    for file in files_list:
        current_file = pq.ParquetFile(file)
        for batch in current_file.iter_batches(batch_size=batchsize):
            chunk = batch.to_pandas()
            if auto_nan_filler:
                for col in cat_cols:
                    if col in chunk.columns:
                        # Вариант 1: Заменить на строку 'NA' или 'missing'
                        chunk[col] = chunk[col].fillna('NA').astype(str)
            
            X_batch = chunk[feature_cols]
            y_batch = chunk[target_col]
            train_pool = Pool(X_batch, y_batch, cat_features=cat_cols)
            
            if first_batch_flag:
                model.fit(train_pool)
                first_batch_flag = False
            else:
                model.fit(train_pool, init_model=model)
    
    model.save_model('../models/catboost_model.cbm')

In [3]:
valid_train_data_path = '../datasets/joined/train_data.parquet'

catboost_model = CatBoostClassifier()

train_in_chunks(files_list=[valid_train_data_path],
                cols_json_file='../datasets/joined/columns_list.json',
                model=catboost_model
                )

Learning rate set to 0.037023
0:	learn: 0.5724585	total: 327ms	remaining: 5m 27s
1:	learn: 0.4748171	total: 388ms	remaining: 3m 13s
2:	learn: 0.3936649	total: 440ms	remaining: 2m 26s
3:	learn: 0.3269747	total: 479ms	remaining: 1m 59s
4:	learn: 0.2721708	total: 624ms	remaining: 2m 4s
5:	learn: 0.2251548	total: 776ms	remaining: 2m 8s
6:	learn: 0.1881843	total: 848ms	remaining: 2m
7:	learn: 0.1582759	total: 903ms	remaining: 1m 51s
8:	learn: 0.1338265	total: 1.01s	remaining: 1m 51s
9:	learn: 0.1137692	total: 1.05s	remaining: 1m 43s
10:	learn: 0.0972436	total: 1.1s	remaining: 1m 38s
11:	learn: 0.0836119	total: 1.13s	remaining: 1m 33s
12:	learn: 0.0721189	total: 1.2s	remaining: 1m 30s
13:	learn: 0.0627904	total: 1.27s	remaining: 1m 29s
14:	learn: 0.0550348	total: 1.3s	remaining: 1m 25s
15:	learn: 0.0485105	total: 1.33s	remaining: 1m 22s
16:	learn: 0.0430376	total: 1.38s	remaining: 1m 20s
17:	learn: 0.0384238	total: 1.43s	remaining: 1m 18s
18:	learn: 0.0344647	total: 1.54s	remaining: 1m 19s
1

CatBoostError: catboost/private/libs/target/target_converter.cpp:404: Target contains only one unique value

## Prediction

In [ ]:
def save_file_results(id_events: pd.Series,
                      predictions: np.ndarray,
                      save_path: str,
                      chunk_num: int):
    """
    Сохранение предсказаний вместе с идентификаторами в Parquet файл
    Принимает:
    id_events - объект pd.Series, который формируется в функции predict_in_chunks.
        Содержит данные об id операций. Нужен для коммита
    predictions - массив нампая с предсказаниями
    save_path - полный путь сохранения parquet файла
    chunk_num - флаговая переменная для указания номера чанка.
        Если чанк под номером 0 (первый), то надо создать новую таблицу. Иначе выполнить соединение с существующей
    """
    result_df = pd.DataFrame({
        'event_id': id_events,
        'predict': predictions
    })
    
    # Конвертируем в Arrow Table
    table = pa.Table.from_pandas(result_df)
    
    if chunk_num == 0:
        # Первый чанк или файл не существует - создаем новый
        pq.write_table(
            table,
            save_path,
            compression='snappy',
            flavor='spark'  # для совместимости со Spark если нужно
        )
        print(f"Создан файл: {save_path} ({len(result_df)} записей)")
    else:
        # Добавляем данные в существующий файл
        with pq.ParquetWriter(
            save_path,
            table.schema,
            compression='snappy',
            flavor='spark',
            append=True  # Важно: append режим
        ) as writer:
            writer.write_table(table)
        
        print(f"Добавлено {len(result_df)} записей в {save_path}")

In [ ]:
def predict_in_chunks(file: str = None,
                      batchsize: int = 20000,
                      cols_json_file: str = None,
                      return_type: str = 'class',  # 'proba' или 'class'
                      model: CatBoostClassifier = None,
                      save_path: List[str] = None
                      ):
    """
    Функция для предсказания на больших файлах по чанкам. Принимает:
    file - файл предсказания (полный путь)
    batchsize - размер чанка
    cols_json_file - путь до json файла с инфой по колонкам
    return_type - 'proba' для вероятностей, 'class' для классов
    model - используемая для предсказания модель
    save_path - полный путь для сохранения
    """
    
    if file is None:
        raise AttributeError("Надо указать file - файл для предсказания")
    if cols_json_file is None:
        raise AttributeError("Надо указать cols_json_file - json файл с инфой по колонкам")
    if model is None:
        raise AttributeError("Надо указать model - действующую модель")
    if save_path is None:
        raise AttributeError("Надо указать путь для сохранения итогового результата")
    
    with open(cols_json_file, 'r') as file:
        cols = json.load(file)
    
    feature_cols = cols['all']
    id_col = cols['event_id']
    
    total_rows = 0

    print(f"Обработка файла: {file}")
    
    current_file = pq.ParquetFile(file)
    
    for batch_idx, batch in enumerate(current_file.iter_batches(batch_size=batchsize)):
        df_chunk = batch.to_pandas()
        chunk = df_chunk[feature_cols]
        id_events=df_chunk[id_col]
        
        if return_type == 'proba':
            preds = model.predict_proba(chunk)[:, 1]
        else:
            preds = model.predict(chunk)
        
        save_file_results(id_events, preds, save_path, chunk_num=batch_idx)
        
        total_rows += len(chunk) 
        if batch_idx % 10 == 0:
            print(f"Обработано чанков: {batch_idx + 1}, строк: {total_rows}")

    print(f"Результаты сохранены в {save_path}")

In [ ]:
valid_predict_saving_path = '../datasets/validation/validation_result.parquet'
valid_predict_data_path = '../datasets/joined/valid_data.parquet'
predict_in_chunks(file=valid_predict_data_path,
                  cols_json_file='../datasets/joined/columns_list.json',
                  model=catboost_model,
                  save_path=valid_predict_saving_path)

## Error Analysis

In [ ]:
# Кастомная функция потерь, которая будет сильно штрафовать за ложно-отрицательные резы

In [ ]:
# Кастомная реализация sklearn.average_precision_score

In [ ]:
def target_class_mistakes_finder(predict_path: str,
                                 answers_path: str,
                                 predict_column: str = 'target',
                                 answers_column: str = 'target',
                                 id_column: str = 'event_id', 
                                 target_class: int = 1):
    """
    Функция по поиску расхождений предикта с известными данными
    именно с целевым классом - False Negative результаты
    Принимает:
    predict_path - полный путь к parquet файлу предсказания
    answers_path - полный путь к parquet файлу с ответами
    predcit_column - название колонки с лейблами предсказаний
    answers_column - название колонки с лейблами ответов
    id_column - название колонки для вывода
    target_class - фильтр по классу
    """
    # Создаем Spark сессию
    spark = SparkSession.builder \
        .appName("FindFalseNegatives") \
        .config("spark.sql.adaptive.enabled", "true") \
        .getOrCreate()

    try:
        # Загружаем данные и проверяем колонки
        predictions = spark.read.parquet(predict_path)
        answers = spark.read.parquet(answers_path)
        
        predict_cols = predictions.columns
        answers_cols = answers.columns
        
        if id_column not in predict_cols:
            raise ValueError(f"В файле предсказаний отсутствует колонка: {id_column}")
        if id_column not in answers_cols:
            raise ValueError(f"В файле ответов отсутствует колонка: {id_column}")
        if predict_column not in predict_cols:
            raise ValueError(f"В файле предсказаний отсутствует колонка: {predict_column}")
        if answers_column not in answers_cols:
            raise ValueError(f"В файле ответов отсутствует колонка: {answers_column}")
        
        # Объединяем данные
        merged = (predictions
                  .select(id_column, predict_column)
                  .join(answers.select(id_column, answers_column),
                        on=id_column,
                        how="inner")
                  )
        
        # Определяем условие для поиска ошибок и находим ошибки
        if target_class in [0, 1, -1]:
            # реальный ответ = target class, предсказание иное
            error_condition = (col(answers_column) == target_class) & (col(predict_column) != target_class)
        else:
            raise ValueError("target_class должен быть 0, 1 или -1")
        
        errors = merged.filter(error_condition)
        
        # Считаем статистику
        total_count = merged.count()
        mismatch_count = merged.filter(col(predict_column) != col(answers_column)).count()
        error_count = errors.count()
        
        print(f"Всего записей: {total_count}")
        print(f"Всего несоответствий: {mismatch_count}")
        print(f"Найдено целевых ошибок: {error_count}")
        
        if error_count > 0:
            print(f"\nПервые 10 {id_column} с ошибками:")
            errors.select(id_column).show(10)

        return [row[id_column] for row in errors.select(id_column).collect()]
        
    finally:
        spark.stop()
    

In [ ]:
# Поиск лучших параметров и отбор полезных признаков (shap?)

In [ ]:
target_class_mistakes_finder(predict_path=valid_predict_saving_path,
                             answers_path=valid_predict_data_path)

## Test Predict (подготовка решения)

In [ ]:
train_files = [valid_train_data_path, valid_predict_data_path]

train_in_chunks(files_list=train_files,
                cols_json_file='../datasets/joined/columns_list.json',
                model=catboost_model)

predict_in_chunks(file='../datasets/test/test.parquet',
                  cols_json_file='../datasets/joined/columns_list.json',
                  model=catboost_model,
                  save_path='../submits/submit.parquet')

## Submit Creating (Создание сабмита)

In [ ]:
save_submit_path = '../submits/submit.csv'

def create_submit(save_path: str = save_submit_path,
                  take_path: str = '../submits/submit.parquet'):
    result_df = pd.read_parquet(take_path)
    
    row_count = len(result_df)
    if row_count != 633683:
        raise ValueError(f"Представленный датафрейм имеет иное количество строк: {row_count}. Такой сабмит не пройдет. Необходимо 633683 строк")
    
    if list(result_df.columns) != ['event_id', 'predict']:
        raise ValueError(f'Неверные названия столбиков: {result_df.columns}')
    
    with open(save_path, 'w') as file:
        result_df.to_csv(file, sep=',', index=False)

In [ ]:
create_submit()